# 25 -- Visualisation summary (v2: dissertation-figure quality)

**Redesigned from v1.** Anything that was a bar chart of a single already-
tabulated number (P1 vs P2, hijack rates, stress-test summaries, device
groups, cohort shift magnitude, per-participant AUC) is now a plain
printed table in Section 2 -- a table communicates one number better than
a bar does, and takes no space to get right.

**What's actually visual now** is everything that shows structure a table
can't: the shape of the learned embedding space (t-SNE), how genuine and
impostor scores actually separate (distributions, not just the AUC
summarising them), real ROC curves, per-seed negative-control spread (not
just a mean+std bar), and the hijack detection time-series traces.

**This needs live model loading and recomputation** (t-SNE, distance
distributions, ROC curves, hijack traces are not pre-exported anywhere) --
unlike v1, which only read existing CSVs. Treat this as new, unexecuted
code with the same first-run caution as nb18-24.


In [ ]:
from pathlib import Path
import json as _json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.manifold import TSNE

%config InlineBackend.close_figures = False
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)
mpl.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.2,
                      'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold'})

def find(*cands):
    for c in cands:
        if Path(c).exists(): return c
    raise FileNotFoundError(cands)

MOD_DIR = Path(find('data/processed/modelling', '../data/processed/modelling', '.'))
DATA_DIR = MOD_DIR.parent
OUT_DIR = DATA_DIR / 'visualisation_summary'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_csv(path, label):
    p = Path(path)
    if not p.exists():
        print(f"SKIP: {label} -- {p} not found")
        return None
    return pd.read_csv(p)

def load_json(path, label):
    p = Path(path)
    if not p.exists():
        print(f"SKIP: {label} -- {p} not found")
        return None
    return _json.load(open(p))

CORE_MODALITIES = ['tap', 'gesture', 'motion']
COLORS = {'tap': '#2b6cb0', 'gesture': '#c0392b', 'motion': '#27ae60', 'typing': '#8e44ad'}


## 1. Load frozen models and embeddings (reused across every visual section below)

In [ ]:
class Embedder(nn.Module):
    def __init__(self, n_features, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(128, embed_dim),
        )
    def forward(self, x):
        z = self.net(x)
        return z / z.norm(dim=1, keepdim=True).clamp_min(1e-8)

def load_modality(m):
    mdir = Path(find(str(DATA_DIR / f'{m}_embedder'), f'{m}_embedder'))
    ckpt = torch.load(mdir / f'{m}_model_seed42.pt', weights_only=False)
    feats = list(ckpt['feature_names'])
    model = Embedder(len(feats), ckpt['embed_dim']); model.load_state_dict(ckpt['state_dict']); model.eval()
    prep = np.load(mdir / f'{m}_preprocessing.npz', allow_pickle=True)
    df = pd.read_parquet(find(str(MOD_DIR / f'{m}_windows.parquet'), f'{m}_windows.parquet'))
    splits = pd.read_csv(find(str(MOD_DIR / 'identity_splits.csv'), 'identity_splits.csv'))
    df = df.merge(splits[['sessionId', 'role']], on='sessionId', how='inner').reset_index(drop=True)
    X = df[feats].to_numpy(dtype=np.float64)
    nan_mask = np.isnan(X)
    if nan_mask.any():
        X = np.where(nan_mask, prep['impute_values'], X)
    X = (X - prep['scale_mean']) / prep['scale_scale']
    with torch.no_grad():
        E = model(torch.tensor(X, dtype=torch.float32)).numpy()
    return df, E

dfs, embeddings = {}, {}
for m in CORE_MODALITIES:
    dfs[m], embeddings[m] = load_modality(m)
    print(f"{m:8s}: {embeddings[m].shape}")

FIXED_WEIGHT_BASIS = {}
for m in CORE_MODALITIES:
    p = DATA_DIR / f'{m}_embedder' / f'{m}_negative_control_summary.csv'
    gap = float(pd.read_csv(p)['gap'].mean()) if p.exists() else 1.0
    FIXED_WEIGHT_BASIS[m] = max(gap, 1e-3)


## 2. Summary tables (was bar charts in v1 -- a table is the right format for one number per row)

In [ ]:
print("=== Per-modality headline (real AUC / shuffled AUC / gap, k=20) ===")
rows = []
for m in ['tap', 'gesture', 'motion', 'typing']:
    man = load_json(DATA_DIR / f'{m}_embedder' / f'{m}_manifest.json', f'{m} manifest')
    if man is None: continue
    real, shuf = man['auc_real_fixed_at_k1_k10_k20'], man['auc_shuffled_fixed_at_k1_k10_k20']
    rows.append({'modality': m, 'n_fixed_identities': man['n_fixed_identities'],
                 'auc_k20': real.get('20'), 'shuffled_auc_k20': shuf.get('20'),
                 'gap': (real.get('20', np.nan) - shuf.get('20', np.nan))})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:.3f}'))

print("\n=== P1 vs P2 ===")
print(pd.DataFrame([
    {'system': 'P1 (single modality, browser)', 'auc': 0.920, 'std_or_note': '+/-0.013'},
    {'system': 'P2 (best fusion)', 'auc': (load_json(DATA_DIR/'fusion_evaluation'/'fusion_manifest.json','fusion')
             or {}).get('best_config_at_k20', {}).get('auc'), 'std_or_note': ''},
]).to_string(index=False))

print("\n=== Fusion sweep, k=20, ranked ===")
fh = load_csv(DATA_DIR / 'fusion_evaluation' / 'fusion_headline_k20.csv', 'fusion headline')
if fh is not None:
    print(fh[['combo','weight_scheme','auc','eer']].to_string(index=False, float_format=lambda v: f'{v:.3f}'))

print("\n=== Hijack detection ===")
hm = load_json(DATA_DIR / 'hijack_evaluation' / 'hijack_manifest.json', 'hijack')
if hm: print(f"  detection rate: {hm['detection_rate']:.1%}, false positive rate: {hm['false_positive_rate']:.1%}")

print("\n=== Stress tests ===")
sm = load_json(DATA_DIR / 'stress_tests' / 'stress_test_manifest.json', 'stress')
if sm:
    print(f"  pace permutation: baseline={sm['test_a_pace_permutation']['baseline_auc']:.3f}, "
          f"shuffled={sm['test_a_pace_permutation']['pace_shuffled_auc']:.3f}")
    print(f"  gradual hijack detection: {sm['test_b_gradual_hijack']['detection_rate']:.1%}")
    print("  modality dropout:"); print(pd.DataFrame(sm['test_c_modality_dropout']).to_string(index=False))
    print("  enrolment ablation:"); print(pd.DataFrame(sm['test_d_enrollment_ablation']).to_string(index=False))
    print(f"  replay sanity check: {sm['test_e_replay_sanity_check']['auc']:.3f}")

print("\n=== Gate 3 device groups ===")
g3 = load_json(DATA_DIR / 'gate3_device_evaluation' / 'gate3_manifest.json', 'gate3')
if g3: print(pd.DataFrame(g3['device_group_results']).to_string(index=False, float_format=lambda v: f'{v:.3f}'))

print("\n=== Cohort shift magnitude (Gate 1 flagged in bold via *) ===")
sma = load_csv(DATA_DIR / 'anomalous_diagnostic' / 'all_identities_shift_magnitude.csv', 'shift magnitude')
if sma is not None:
    FLAGGED = {'pUNKH7L','pAFQRTM','pEAB9GS'}
    sma = sma.sort_values('shift_magnitude', ascending=False).copy()
    sma['flagged'] = sma['participantId'].isin(FLAGGED).map({True:'*', False:''})
    print(sma.to_string(index=False, float_format=lambda v: f'{v:.2f}'))

print("\n=== Task generalisation (tap only, see nb24 for full discussion) ===")
tg = load_csv(DATA_DIR / 'task_generalisation' / 'task_generalisation_results.csv', 'task gen')
if tg is not None:
    print(tg.to_string(index=False, float_format=lambda v: f'{v:.3f}'))
    print("Not re-visualised here as a distribution -- would require retraining Model A again")
    print("just for a nicer plot, which nb24 already answered definitively as a table.")


## 3. Embedding space structure (t-SNE) -- does the learned space actually organise by person?

The direct visual analogue of P1's Section 9.2 finding ("the learned
representation organises itself by person"), reproduced here for P2's
trained embedders. Coloured by identity; probe-role points are marked
with a black outline so you can see whether a person's TEST session
lands inside their own cluster or elsewhere.


In [ ]:
fig, axes = plt.subplots(1, len(CORE_MODALITIES), figsize=(7 * len(CORE_MODALITIES), 6.5))
for ax, m in zip(axes, CORE_MODALITIES):
    df, E = dfs[m], embeddings[m]
    n = len(E)
    perplexity = min(30, max(5, n // 20))
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, init='pca')
    coords = tsne.fit_transform(E)

    pids = df['participantId'].values
    unique_pids = sorted(set(pids))
    cmap = plt.get_cmap('tab20', len(unique_pids))
    pid_color = {p: cmap(i) for i, p in enumerate(unique_pids)}

    is_probe = (df['role'] == 'probe').values
    ax.scatter(coords[~is_probe, 0], coords[~is_probe, 1],
               c=[pid_color[p] for p in pids[~is_probe]], s=18, alpha=0.5, linewidths=0)
    ax.scatter(coords[is_probe, 0], coords[is_probe, 1],
               c=[pid_color[p] for p in pids[is_probe]], s=55, alpha=0.95,
               edgecolors='black', linewidths=1.0, label='probe windows')
    ax.set_title(f'{m}: t-SNE of learned embeddings\n(outlined = probe/test windows)')
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
fig.savefig(OUT_DIR / '03_tsne_embedding_space.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Genuine vs. impostor distance distributions -- the shape behind the AUC number

Per modality: distance from every probe window to its OWN reference
(genuine) vs. to every OTHER candidate's reference (impostor). This is
the direct visual of what AUC summarises into one number -- the overlap
between these two distributions IS the error rate.


In [ ]:
def genuine_impostor_distances(m):
    df, E = dfs[m], embeddings[m]
    enrol_mask, probe_mask = (df['role']=='enrol').values, (df['role']=='probe').values
    pid_arr = df['participantId'].values
    refs = {pid: E[enrol_mask & (pid_arr==pid)].mean(axis=0) for pid in np.unique(pid_arr[enrol_mask])}
    genuine, impostor = [], []
    for i in np.where(probe_mask)[0]:
        true_pid = pid_arr[i]
        for cand, ref in refs.items():
            d = float(np.linalg.norm(E[i] - ref))
            (genuine if cand == true_pid else impostor).append(d)
    return np.array(genuine), np.array(impostor)

fig, axes = plt.subplots(1, len(CORE_MODALITIES), figsize=(6.5 * len(CORE_MODALITIES), 5))
for ax, m in zip(axes, CORE_MODALITIES):
    genuine, impostor = genuine_impostor_distances(m)
    bins = np.linspace(0, max(genuine.max(), impostor.max()), 40)
    ax.hist(impostor, bins=bins, alpha=0.55, color='#c0392b', density=True, label=f'impostor (n={len(impostor)})')
    ax.hist(genuine, bins=bins, alpha=0.65, color='#2b6cb0', density=True, label=f'genuine (n={len(genuine)})')
    ax.axvline(genuine.mean(), color='#2b6cb0', ls='--', lw=1.5)
    ax.axvline(impostor.mean(), color='#c0392b', ls='--', lw=1.5)
    ax.set_xlabel('distance to reference'); ax.set_ylabel('density')
    ax.set_title(f'{m}: genuine vs. impostor distance')
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / '04_genuine_impostor_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. ROC curves -- per modality and best fusion, overlaid

Session-level aggregation (one score per probe session per candidate,
same method as nb20/21/24) rather than single-window, for a stable
curve. Fusion uses the fixed-weight scheme with each modality's own
already-computed reliability basis.


In [ ]:
def fused_session_score(modalities, weight_basis):
    session_z = {}
    for m in modalities:
        df, E = dfs[m], embeddings[m]
        enrol_mask, probe_mask = (df['role']=='enrol').values, (df['role']=='probe').values
        pid_arr, sid_arr = df['participantId'].values, df['sessionId'].values
        refs = {pid: E[enrol_mask & (pid_arr==pid)].mean(axis=0) for pid in np.unique(pid_arr[enrol_mask])}
        sess_positions = {}
        for i in np.where(probe_mask)[0]:
            sess_positions.setdefault(sid_arr[i], []).append(i)
        rows = []
        for sid, idxs in sess_positions.items():
            true_pid = pid_arr[idxs[0]]
            agg_E = E[idxs].mean(axis=0)
            for cand, ref in refs.items():
                rows.append({'sessionId': sid, 'pid': true_pid, 'cand': cand,
                             'genuine': int(cand==true_pid), 'dist': float(np.linalg.norm(agg_E-ref))})
        rdf = pd.DataFrame(rows)
        rdf['z'] = rdf.groupby('sessionId')['dist'].transform(lambda s: (s-s.mean())/(s.std() if s.std()>0 else 1))
        session_z[m] = rdf.set_index(['sessionId','cand'])['z']
    all_keys = set()
    for s in session_z.values(): all_keys |= set(s.index)
    fused = pd.DataFrame(list(all_keys), columns=['sessionId','cand'])
    fused['wz_sum'], fused['w_sum'] = 0.0, 0.0
    for m in modalities:
        s = session_z[m]
        idx = pd.MultiIndex.from_frame(fused[['sessionId','cand']])
        vals = pd.Series(idx.map(lambda k: s.get(k, np.nan)), index=fused.index)
        present = vals.notna()
        fused.loc[present,'wz_sum'] += vals[present]*weight_basis[m]
        fused.loc[present,'w_sum'] += weight_basis[m]
    fused = fused[fused['w_sum']>0].copy()
    fused['combined_z'] = fused['wz_sum']/fused['w_sum']
    sid_to_pid = dfs[modalities[0]].drop_duplicates('sessionId').set_index('sessionId')['participantId']
    fused['pid'] = fused['sessionId'].map(sid_to_pid)
    fused = fused.dropna(subset=['pid'])
    fused['genuine'] = (fused['pid']==fused['cand']).astype(int)
    return fused['genuine'].values, -fused['combined_z'].values

fig, ax = plt.subplots(figsize=(7.5, 7))
for m in CORE_MODALITIES:
    genuine, impostor = genuine_impostor_distances(m)
    y_true = np.concatenate([np.ones(len(genuine)), np.zeros(len(impostor))])
    scores = -np.concatenate([genuine, impostor])
    fpr, tpr, _ = roc_curve(y_true, scores)
    auc = roc_auc_score(y_true, scores)
    ax.plot(fpr, tpr, color=COLORS[m], lw=2, label=f'{m} (AUC={auc:.3f})')

y_true_fused, scores_fused = fused_session_score(CORE_MODALITIES, FIXED_WEIGHT_BASIS)
fpr_f, tpr_f, _ = roc_curve(y_true_fused, scores_fused)
auc_f = roc_auc_score(y_true_fused, scores_fused)
ax.plot(fpr_f, tpr_f, color='black', lw=2.5, ls='--', label=f'fusion, all 3 (AUC={auc_f:.3f})')

ax.plot([0,1],[0,1], color='gray', ls=':', lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC curves: per-modality vs. fusion')
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()
fig.savefig(OUT_DIR / '05_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Negative control: per-seed spread, not just mean +/- std

Shows the actual distribution of shuffled-label AUC across seeds
(stripplot-style, one point per seed) against the real-label AUC as a
reference line -- makes the instability this project spent real time
diagnosing directly visible, rather than compressed into an error bar.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5.5), sharey=True)
for ax, m in zip(axes, ['tap','gesture','motion','typing']):
    raw = load_csv(DATA_DIR / f'{m}_embedder' / f'{m}_negative_control_raw.csv', f'{m} neg control raw')
    man = load_json(DATA_DIR / f'{m}_embedder' / f'{m}_manifest.json', f'{m} manifest')
    if raw is None or man is None:
        ax.set_visible(False); continue
    sub20 = raw[raw['k']==20]
    rng = np.random.default_rng(0)
    jitter = rng.uniform(-0.08, 0.08, len(sub20))
    ax.scatter(np.zeros(len(sub20)) + jitter, sub20['auc'], s=50, alpha=0.7, color=COLORS[m])
    real_auc = man['auc_real_fixed_at_k1_k10_k20'].get('20')
    if real_auc: ax.axhline(real_auc, color='black', ls='--', lw=1.5, label=f'real AUC={real_auc:.3f}')
    ax.axhline(0.5, color='gray', ls=':', lw=1)
    ax.set_xlim(-0.5, 0.5); ax.set_xticks([])
    ax.set_title(f'{m}\n(n={len(sub20)} seeds)')
    ax.legend(fontsize=8)
axes[0].set_ylabel('AUC at k=20')
fig.suptitle('Negative control: per-seed shuffled AUC spread vs. real AUC', y=1.02, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_DIR / '06_negative_control_spread.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. AUC vs. trust window k -- all modalities + fusion

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5))
for m in ['tap','gesture','motion','typing']:
    df_tvb = load_csv(DATA_DIR / f'{m}_embedder' / f'{m}_trained_vs_baseline.csv', f'{m} trained_vs_baseline')
    if df_tvb is None: continue
    sub = df_tvb[df_tvb['identity_set']=='fixed_across_k']
    if sub.empty: sub = df_tvb[df_tvb['identity_set']=='all_available']
    ax.plot(sub['k'], sub['auc'], marker='o', color=COLORS[m], label=m, lw=2)

sweep = load_csv(DATA_DIR / 'fusion_evaluation' / 'fusion_sweep_full.csv', 'fusion sweep')
fh = load_csv(DATA_DIR / 'fusion_evaluation' / 'fusion_headline_k20.csv', 'fusion headline')
if sweep is not None and fh is not None and len(fh):
    best_combo, best_scheme = fh.iloc[0]['combo'], fh.iloc[0]['weight_scheme']
    sub = sweep[(sweep['combo']==best_combo)&(sweep['weight_scheme']==best_scheme)&
                (sweep['identity_set']=='fixed_across_k')].sort_values('k')
    ax.plot(sub['k'], sub['auc'], marker='s', color='black', lw=2.5, ls='--', label=f'fusion ({best_combo})')

ax.axhline(0.5, color='gray', ls=':', lw=1)
ax.set_xlabel('k (trust window)'); ax.set_ylabel('AUC')
ax.set_title('Verification AUC vs. trust window, all modalities + best fusion')
ax.legend()
plt.tight_layout()
fig.savefig(OUT_DIR / '07_auc_vs_k.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Hijack detection traces -- small multiples: real detections vs. a false-positive control

Reconstructs the same splice/detection logic from nb19/nb20 (self-
contained here, not re-importing another notebook) to show a few example
traces side by side: what a caught switch looks like, and what a genuine,
un-spliced session that STILL gets wrongly flagged looks like -- the
honest limitation, shown directly rather than only stated as a rate.


In [ ]:
ROLL_W, DETECT_SIGMA, DEBOUNCE_N, CALIBRATION_FRACTION = 5, 3.0, 5, 0.5

def get_probe_sessions(pid, m):
    d = dfs[m]
    rows = d[(d['role']=='probe') & (d['participantId']==pid)]
    return {sid: g.sort_values('window_index').index.to_numpy() for sid, g in rows.groupby('sessionId')}

refs_by_m = {m: {pid: embeddings[m][(dfs[m]['role']=='enrol').values & (dfs[m]['participantId']==pid).values].mean(axis=0)
                 for pid in dfs[m][dfs[m]['role']=='enrol']['participantId'].unique()} for m in CORE_MODALITIES}

def build_trace(pid_a, pid_b, split_fraction=0.5):
    trace = {}
    for m in CORE_MODALITIES:
        sess_a, sess_b = get_probe_sessions(pid_a, m), get_probe_sessions(pid_b, m)
        if not sess_a or not sess_b: continue
        a_idx, b_idx = list(sess_a.values())[0], list(sess_b.values())[0]
        if len(a_idx) < 4 or len(b_idx) < 4: continue
        split_at = max(2, int(len(a_idx)*split_fraction))
        ref_a = refs_by_m[m].get(pid_a)
        if ref_a is None: continue
        combined = np.concatenate([a_idx[:split_at], b_idx])
        dist = np.linalg.norm(embeddings[m][combined] - ref_a, axis=1)
        trace[m] = {'dist': dist, 'split_index': split_at}
    return trace if trace else None

def fuse_trace(trace):
    n = max(len(v['dist']) for v in trace.values())
    split_index = min(v['split_index'] for v in trace.values())
    calib_end = max(2, int(split_index*CALIBRATION_FRACTION))
    zs = {}
    for m, v in trace.items():
        calib = v['dist'][:calib_end]
        mu, sigma = calib.mean(), (calib.std() if calib.std()>0 else 1.0)
        zs[m] = (v['dist']-mu)/sigma
    fused = np.full(n, np.nan)
    for i in range(n):
        vals, w = [], []
        for m, z in zs.items():
            if i < len(z): vals.append(z[i]); w.append(FIXED_WEIGHT_BASIS[m])
        if vals: fused[i] = np.average(vals, weights=w)
    rolled = pd.Series(fused).rolling(ROLL_W, min_periods=ROLL_W).mean().to_numpy()
    return rolled, split_index

def detect(rolled, split_index, calib_end):
    baseline = rolled[:calib_end]; baseline = baseline[~np.isnan(baseline)]
    if len(baseline) < 2: return None, np.nan
    thresh = baseline.mean() + DETECT_SIGMA*baseline.std()
    run = 0
    for i in range(split_index, len(rolled)):
        if np.isnan(rolled[i]): run = 0; continue
        run = run+1 if rolled[i] > thresh else 0
        if run >= DEBOUNCE_N: return i-DEBOUNCE_N+1, thresh
    return None, thresh

usable_pids = sorted(set(dfs['tap'][dfs['tap']['role']=='probe']['participantId']))
example_pairs, controls = [], []
for a in usable_pids:
    for b in usable_pids:
        if a == b: continue
        t = build_trace(a, b)
        if t is None: continue
        rolled, si = fuse_trace(t)
        ce = max(2, int(si*CALIBRATION_FRACTION))
        idx, thresh = detect(rolled, si, ce)
        if idx is not None and idx >= si and len(example_pairs) < 2:
            example_pairs.append((a, b, rolled, si, thresh))
    if len(example_pairs) >= 2: break

for a in usable_pids:
    t = build_trace(a, a)
    if t is None: continue
    rolled, si = fuse_trace(t)
    ce = max(2, int(si*CALIBRATION_FRACTION))
    idx, thresh = detect(rolled, si, ce)
    if idx is not None and idx >= si:
        controls.append((a, rolled, si, thresh)); break

n_panels = len(example_pairs) + len(controls)
if n_panels:
    fig, axes = plt.subplots(1, n_panels, figsize=(6*n_panels, 5), sharey=True)
    axes = np.atleast_1d(axes)
    panel = 0
    for a, b, rolled, si, thresh in example_pairs:
        ax = axes[panel]; panel += 1
        ax.plot(rolled, marker='o', ms=3, lw=1, color='#2b6cb0')
        ax.axvline(si, color='red', ls='--', label='splice')
        ax.axhline(thresh, color='gray', ls=':', label='threshold')
        ax.set_title(f'DETECTED: {a} -> {b}'); ax.legend(fontsize=7)
    for a, rolled, si, thresh in controls:
        ax = axes[panel]; panel += 1
        ax.plot(rolled, marker='o', ms=3, lw=1, color='#c0392b')
        ax.axvline(si, color='red', ls='--', label='(no actual splice)')
        ax.axhline(thresh, color='gray', ls=':', label='threshold')
        ax.set_title(f'FALSE POSITIVE: {a} (self-continuation)'); ax.legend(fontsize=7)
    axes[0].set_ylabel('fused rolling z-distance to reference')
    for ax in axes: ax.set_xlabel('window position')
    plt.tight_layout()
    fig.savefig(OUT_DIR / '08_hijack_traces.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No example traces found -- check that CORE_MODALITIES have enough probe data.")


## 9. `pAFQRTM` feature-shift fingerprint, grouped by mechanism

Same top-shifted-features data as nb13's diagnosis, regrouped by feature
FAMILY (device-orientation coupling vs. accelerometer coupling) rather
than a flat ranked list -- visually shows the shift is concentrated in
ONE mechanism (device-motion response to a tap), not scattered, which is
the actual evidence for "this is a real behavioural change" over "this is
noise".


In [ ]:
fs_path = DATA_DIR / 'anomalous_diagnostic' / 'pAFQRTM_feature_shift.csv'
fs = load_csv(fs_path, 'pAFQRTM feature shift')
if fs is not None:
    def family(name):
        if 'orient' in name: return 'orientation coupling'
        if 'accel' in name: return 'accelerometer coupling'
        if any(k in name for k in ['dwell','ptp','rtp','reaction','interval']): return 'timing'
        return 'other'
    fs['family'] = fs['feature'].apply(family)
    top = fs.reindex(fs['standardised_shift'].abs().sort_values(ascending=False).index).head(15)
    family_colors = {'orientation coupling': '#2b6cb0', 'accelerometer coupling': '#27ae60',
                      'timing': '#c0392b', 'other': '#7f8c8d'}
    fig, ax = plt.subplots(figsize=(9, 7))
    colors = [family_colors[f] for f in top['family']]
    ax.barh(range(len(top)), top['standardised_shift'], color=colors)
    ax.set_yticks(range(len(top))); ax.set_yticklabels(top['feature'], fontsize=8)
    ax.invert_yaxis()
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('standardised shift, train -> enrol')
    ax.set_title('pAFQRTM: feature shift by mechanism\n(grip-change signature: motion-coupling, not timing)')
    handles = [plt.Rectangle((0,0),1,1, color=c) for c in family_colors.values()]
    ax.legend(handles, family_colors.keys(), fontsize=8, loc='lower right')
    plt.tight_layout()
    fig.savefig(OUT_DIR / '09_pAFQRTM_feature_fingerprint.png', dpi=150, bbox_inches='tight')
    plt.show()


## Index

In [ ]:
print(f"Plots written to {OUT_DIR.resolve()}:")
for p in sorted(OUT_DIR.glob('*.png')):
    print(f"  {p.name}")
